<a href="https://colab.research.google.com/github/sulucay01/DI725-assignment1/blob/dev/notebooks/03_modeling_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Weights & Biases: https://wandb.ai/selingoktas98-metu-middle-east-technical-university/di725-assignment1?nw=nwuserselingoktas98

GitHub: https://github.com/sulucay01/DI725-assignment1

# Objective of the notebook

This notebook trains RoBERTa-base models for customer sentiment
classification using the processed conversation text.

Main goals:
1. compare truncation strategies (head, tail, head-tail)
2. establish a strong text-only baseline
3. evaluate several imbalance-handling methods
4. identify a suitable baseline configuration before moving to fusion models

All experiments use the same preprocessing outputs and the same
train-validation split created in the preprocessing notebook.

# Part 1: Truncation Strategy Experiments

In [1]:
# =========================================================
# Imports
# =========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import wandb
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)

In [2]:
# =========================================================
# Global Experiment Configuration
# =========================================================
# Centralize all core experiment settings to ensure consistency
# across runs and make the notebook easier to maintain.
#
# Any change here will affect all experiments in this notebook.

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
PROJECT_NAME = "di725-assignment1"

SPECIAL_TOKENS = ["[CUSTOMER]", "[AGENT]", "[EMAIL]", "[PHONE]", "[ID]"]

In [3]:
# =========================================================
# Reproducibility Setup
# =========================================================
# Fix random seeds and deterministic settings to improve
# reproducibility across experiments.
#
# Since this notebook runs multiple training experiments,
# seeds are set once globally and also reset before each run
# so that model initialization, sampling, and training behavior
# remain as consistent as possible.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Seed set to:", SEED)
print("CUDA available:", torch.cuda.is_available())

# Reset all random seeds before each experiment so that
# model initialization, sampling, and training behavior
# remain consistent across runs.

def reset_all_seeds(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    set_seed(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

Seed set to: 42
CUDA available: True


In [4]:
# =========================================================
# Load Processed Datasets
# =========================================================
# Load the preprocessed train and validation datasets created
# in the preprocessing notebook.
#
# These datasets already include:
# - cleaned conversation text
# - normalized categorical fields
# - encoded sentiment labels

# Note:
# Only train and validation datasets are used in this notebook.
# Test data is intentionally excluded from modeling decisions
# to avoid unintended bias during experimentation.

train_df = pd.read_csv("https://raw.githubusercontent.com/sulucay01/DI725-assignment1/dev/data/processed/train_processed.csv")
val_df = pd.read_csv("https://raw.githubusercontent.com/sulucay01/DI725-assignment1/dev/data/processed/val_processed.csv")

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

Train shape: (773, 13)
Val shape: (194, 13)


In [5]:
# =========================================================
# Validate Expected Columns
# =========================================================
# Check required columns, missing values, and label distribution.

required_cols = ["conversation", "clean_text", "label", "customer_sentiment"]

for name, df in [("train", train_df), ("val", val_df)]:
    print(f"\n{name.upper()}")
    print("Missing required columns:", [c for c in required_cols if c not in df.columns])
    print("Null clean_text:", df["clean_text"].isna().sum())
    print("Empty clean_text:", (df["clean_text"].fillna("").str.strip() == "").sum())
    print("Label distribution:")
    print(df["customer_sentiment"].value_counts(normalize=True))


TRAIN
Missing required columns: []
Null clean_text: 0
Empty clean_text: 0
Label distribution:
customer_sentiment
neutral     0.560155
negative    0.421734
positive    0.018111
Name: proportion, dtype: float64

VAL
Missing required columns: []
Null clean_text: 0
Empty clean_text: 0
Label distribution:
customer_sentiment
neutral     0.561856
negative    0.422680
positive    0.015464
Name: proportion, dtype: float64


In [6]:
# =========================================================
# Tokenizer Setup
# =========================================================
# Load tokenizer and add custom special tokens from preprocessing.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
num_added = tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

# Helper to check sequence lengths before/after truncation.
def get_token_lengths(texts, tokenizer):
    lengths = []
    for text in texts:
        enc = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            padding=False
        )
        lengths.append(len(enc["input_ids"]))
    return lengths

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# Set initial truncation strategy for baseline run.
TRUNCATION_STRATEGY = "head"

# =========================================================
# Manual Truncation
# =========================================================
# Apply head / tail / head-tail truncation before tokenization.

def truncate_text(text, tokenizer, max_length, strategy="head"):
    if not isinstance(text, str):
        text = ""

    token_ids = tokenizer.encode(text, add_special_tokens=False)
    token_budget = max_length - 2  # reserve space for <s> and </s>

    if len(token_ids) <= token_budget:
        return text

    if strategy == "head":
        kept_ids = token_ids[:token_budget]
    elif strategy == "tail":
        kept_ids = token_ids[-token_budget:]
    elif strategy == "head_tail":
        head_size = token_budget // 2
        tail_size = token_budget - head_size
        kept_ids = token_ids[:head_size] + token_ids[-tail_size:]
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return tokenizer.decode(
        kept_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=True
    )

# Truncate text to fit max_length based on selected strategy.
def apply_truncation(df, strategy, tokenizer, max_length):
    out = df.copy()
    out["model_text"] = out["clean_text"].apply(
        lambda x: truncate_text(x, tokenizer, max_length, strategy=strategy)
    )
    return out

## Part 1.1 - Head Truncation

In [8]:
# =========================================================
# Head Truncation Baseline
# =========================================================
# Run the first baseline with head-only truncation.

# Apply truncation to train and validation data.
train_exp = apply_truncation(train_df, TRUNCATION_STRATEGY, tokenizer, MAX_LENGTH)
val_exp = apply_truncation(val_df, TRUNCATION_STRATEGY, tokenizer, MAX_LENGTH)

print(train_exp[["clean_text", "model_text"]].head(2))

Token indices sequence length is longer than the specified maximum sequence length for this model (514 > 512). Running this sequence through the model will result in indexing errors


                                          clean_text  \
0  [AGENT] Hello, thank you for contacting BrownB...   
1  [CUSTOMER] Hi, I need help with returning a to...   

                                          model_text  
0  [AGENT] Hello, thank you for contacting BrownB...  
1  [CUSTOMER] Hi, I need help with returning a to...  


In [9]:
# Check that truncated texts fit within max_length.

train_model_lengths = get_token_lengths(train_exp["model_text"].tolist(), tokenizer)
val_model_lengths = get_token_lengths(val_exp["model_text"].tolist(), tokenizer)

print("Train max length after truncation:", max(train_model_lengths))
print("Val max length after truncation:", max(val_model_lengths))

Train max length after truncation: 512
Val max length after truncation: 512


In [10]:
# =========================================================
# Convert to Hugging Face Dataset
# =========================================================
# Keep only model_text and label for training.

train_hf = Dataset.from_pandas(train_exp[["model_text", "label"]], preserve_index=False)
val_hf = Dataset.from_pandas(val_exp[["model_text", "label"]], preserve_index=False)

print(train_hf)
print(val_hf)

Dataset({
    features: ['model_text', 'label'],
    num_rows: 773
})
Dataset({
    features: ['model_text', 'label'],
    num_rows: 194
})


In [11]:
# =========================================================
# Tokenization
# =========================================================
# Tokenize model_text for RoBERTa input.

def tokenize_batch(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

train_hf = train_hf.map(tokenize_batch, batched=True)
val_hf = val_hf.map(tokenize_batch, batched=True)

# Rename label column to labels for Trainer.
train_hf = train_hf.rename_column("label", "labels")
val_hf = val_hf.rename_column("label", "labels")

columns_to_keep = ["input_ids", "attention_mask", "labels"]

# Keep only the tensors required by the model:
# - input_ids
# - attention_mask
# - labels

train_hf.set_format(type="torch", columns=columns_to_keep)
val_hf.set_format(type="torch", columns=columns_to_keep)

# Use dynamic padding during batching.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_hf[0].keys())
print(len(train_hf[0]["input_ids"]), len(train_hf[0]["attention_mask"]))
print(train_hf[0]["labels"])

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'attention_mask'])
512 512
tensor(0)


In [12]:
# =========================================================
# Label Mapping and Metrics
# =========================================================
# Recover class names and define evaluation metrics.

label_map_df = (
    train_df[["customer_sentiment", "label"]]
    .drop_duplicates()
    .sort_values("label")
)

label_names = label_map_df["customer_sentiment"].tolist()
num_labels = len(label_names)

print("Num labels:", num_labels)
print("Label names in encoded order:", label_names)

# Compute overall and per-class classification metrics.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    metrics = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

    for i in range(len(precision)):
        metrics[f"precision_class_{i}"] = precision[i]
        metrics[f"recall_class_{i}"] = recall[i]
        metrics[f"f1_class_{i}"] = f1[i]

    return metrics

Num labels: 3
Label names in encoded order: ['negative', 'neutral', 'positive']


In [13]:
# =========================================================
# Weights & Biases Setup
# =========================================================
# Initialize experiment tracking and log configuration.

RUN_NAME = f"roberta_{TRUNCATION_STRATEGY}_{MAX_LENGTH}"

wandb.login()

wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "truncation_strategy": TRUNCATION_STRATEGY,
        "text_column": "clean_text",
        "model_text_column": "model_text",
        "num_labels": num_labels,
        "train_size": len(train_df),
        "val_size": len(val_df),
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: selingoktas98 (selingoktas98-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
# =========================================================
# Model Initialization
# =========================================================
# Load RoBERTa model and resize embeddings for new tokens.

reset_all_seeds(SEED)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and

Embedding(50270, 768, padding_idx=1)

In [15]:
# =========================================================
# Training Configuration
# =========================================================
# Define training settings and evaluation strategy.

training_args = TrainingArguments(
    output_dir=f"./outputs/{RUN_NAME}_new",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1", # Select best model based on validation macro F1.
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=RUN_NAME,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
# =========================================================
# Trainer Setup
# =========================================================
# Configure Trainer with model, data, and metrics.

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Print run configuration and data info.
print("Run name:", RUN_NAME)
print("Train size:", len(train_hf))
print("Val size:", len(val_hf))
print("Tokenizer size:", len(tokenizer))
print("Model vocab resized:", model.get_input_embeddings().weight.shape[0])
print("Max train sequence length after truncation:", max(train_model_lengths))

Run name: roberta_head_512
Train size: 773
Val size: 194
Tokenizer size: 50270
Model vocab resized: 50270
Max train sequence length after truncation: 512


In [17]:
# Train model and evaluate on validation set.
train_result = trainer.train()
val_metrics = trainer.evaluate()

print("\nValidation metrics:")
print(val_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2
1,0.555443,0.543516,0.829897,0.549668,0.817583,0.964286,0.658537,0.782609,0.775362,0.981651,0.866397,0.000000,0.000000,0.000000
2,0.394463,0.425625,0.881443,0.589273,0.872891,0.969697,0.780488,0.864865,0.835938,0.981651,0.902954,0.000000,0.000000,0.000000
3,0.176330,0.419907,0.881443,0.590453,0.873915,0.918919,0.829268,0.871795,0.858333,0.944954,0.899563,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation metrics:
{'eval_loss': 0.4199065864086151, 'eval_accuracy': 0.8814432989690721, 'eval_macro_f1': 0.5904527301907215, 'eval_weighted_f1': 0.873915367185074, 'eval_precision_class_0': 0.918918918918919, 'eval_recall_class_0': 0.8292682926829268, 'eval_f1_class_0': 0.8717948717948718, 'eval_precision_class_1': 0.8583333333333333, 'eval_recall_class_1': 0.944954128440367, 'eval_f1_class_1': 0.8995633187772926, 'eval_precision_class_2': 0.0, 'eval_recall_class_2': 0.0, 'eval_f1_class_2': 0.0, 'eval_runtime': 1.3313, 'eval_samples_per_second': 145.719, 'eval_steps_per_second': 9.765, 'epoch': 3.0}


In [18]:
# =========================================================
# Detailed Evaluation
# =========================================================
# Evaluate model performance with reports and confusion matrix.

def evaluate_split(trainer, dataset, split_name, label_names):
    predictions = trainer.predict(dataset) # raw model outputs for the split
    logits = predictions.predictions
    y_true = predictions.label_ids
    y_pred = np.argmax(logits, axis=1) # predicted class ids

    all_labels = list(range(len(label_names)))

    print(f"\n===== {split_name.upper()} CLASSIFICATION REPORT =====")
    print(classification_report(
        y_true,
        y_pred,
        labels=all_labels,
        target_names=label_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    print(f"\n===== {split_name.upper()} CONFUSION MATRIX =====")
    print(cm)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=all_labels,
        average=None,
        zero_division=0
    )

    metrics = {
        f"{split_name}_accuracy": accuracy_score(y_true, y_pred),
        f"{split_name}_macro_f1": f1_score(y_true, y_pred, average="macro"),
        f"{split_name}_weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    for i in range(len(precision)):
        metrics[f"{split_name}_precision_class_{i}"] = precision[i]
        metrics[f"{split_name}_recall_class_{i}"] = recall[i]
        metrics[f"{split_name}_f1_class_{i}"] = f1[i]

    wandb.log(metrics)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "logits": logits,
        "confusion_matrix": cm,
        "metrics": metrics
    }

In [19]:
# Evaluate baseline model on train and validation sets.
train_results = evaluate_split(trainer, train_hf, "train", label_names)
val_results = evaluate_split(trainer, val_hf, "val", label_names)

head_results = {
    "run_name": RUN_NAME,
    "strategy": "head",
    "train_results": train_results,
    "val_results": val_results
}

wandb.finish()


===== TRAIN CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.96      0.91      0.93       326
     neutral       0.91      0.97      0.94       433
    positive       0.00      0.00      0.00        14

    accuracy                           0.93       773
   macro avg       0.62      0.63      0.62       773
weighted avg       0.91      0.93      0.92       773


===== TRAIN CONFUSION MATRIX =====
[[296  30   0]
 [ 13 420   0]
 [  0  14   0]]



===== VAL CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.92      0.83      0.87        82
     neutral       0.86      0.94      0.90       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.88       194
   macro avg       0.59      0.59      0.59       194
weighted avg       0.87      0.88      0.87       194


===== VAL CONFUSION MATRIX =====
[[ 68  14   0]
 [  6 103   0]
 [  0   3   0]]


eval/accuracy,▁███
eval/f1_class_0,▁▇██
eval/f1_class_1,▁█▇▇
eval/f1_class_2,▁▁▁▁
eval/loss,█▁▁▁
eval/macro_f1,▁███
eval/precision_class_0,▇█▁▁
eval/precision_class_1,▁▆██
eval/precision_class_2,▁▁▁▁
eval/recall_class_0,▁▆██
+51,...


In [20]:
# =========================================================
# Reusable Experiment Function
# =========================================================
# Run the full pipeline for a given truncation strategy.

def run_experiment(
    strategy,
    train_df,
    val_df,
    tokenizer,
    model_name,
    max_length,
    seed,
    num_labels,
    label_names,
    project_name="di725-assignment1"
):
    reset_all_seeds(seed)

    # Train and evaluate one truncation strategy.
    run_name = f"roberta_{strategy}_{max_length}"
    print(f"\n===== RUNNING EXPERIMENT: {run_name} =====")

    # Apply truncation to train and validation data.
    train_exp = apply_truncation(train_df, strategy, tokenizer, max_length)
    val_exp = apply_truncation(val_df, strategy, tokenizer, max_length)

    # Check sequence lengths after truncation.
    train_model_lengths = get_token_lengths(train_exp["model_text"].tolist(), tokenizer)
    val_model_lengths = get_token_lengths(val_exp["model_text"].tolist(), tokenizer)

    print("Train max length after truncation:", max(train_model_lengths))
    print("Val max length after truncation:", max(val_model_lengths))

    # Build Hugging Face datasets.
    train_hf = Dataset.from_pandas(train_exp[["model_text", "label"]], preserve_index=False)
    val_hf = Dataset.from_pandas(val_exp[["model_text", "label"]], preserve_index=False)

    # Tokenize model inputs.
    def tokenize_batch_local(batch):
        return tokenizer(
            batch["model_text"],
            truncation=True,
            max_length=max_length,
            padding=False
        )

    train_hf = train_hf.map(tokenize_batch_local, batched=True)
    val_hf = val_hf.map(tokenize_batch_local, batched=True)

    train_hf = train_hf.rename_column("label", "labels")
    val_hf = val_hf.rename_column("label", "labels")

    columns_to_keep = ["input_ids", "attention_mask", "labels"]
    train_hf.set_format(type="torch", columns=columns_to_keep)
    val_hf.set_format(type="torch", columns=columns_to_keep)

    data_collator_local = DataCollatorWithPadding(tokenizer=tokenizer)

    # Initialize W&B run.
    wandb.init(
        project=project_name,
        name=run_name,
        config={
            "seed": seed,
            "model_name": model_name,
            "max_length": max_length,
            "truncation_strategy": strategy,
            "text_column": "clean_text",
            "model_text_column": "model_text",
            "num_labels": num_labels,
            "train_size": len(train_df),
            "val_size": len(val_df),
        }
    )

    # Load a fresh model for a fair comparison.
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    model.resize_token_embeddings(len(tokenizer))

    training_args = TrainingArguments(
        output_dir=f"./outputs/{run_name}_new",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=20,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="wandb",
        run_name=run_name,
        fp16=False,
        seed=seed,
        data_seed=seed,
        dataloader_num_workers=0
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_hf,
        eval_dataset=val_hf,
        data_collator=data_collator_local,
        compute_metrics=compute_metrics
    )

    # Train and evaluate.
    trainer.train()
    val_metrics = trainer.evaluate()

    print("\nValidation metrics:")
    print(val_metrics)

    train_results = evaluate_split(trainer, train_hf, "train", label_names)
    val_results = evaluate_split(trainer, val_hf, "val", label_names)

    wandb.finish()

    return {
        "run_name": run_name,
        "strategy": strategy,
        "val_metrics": val_metrics,
        "train_results": train_results,
        "val_results": val_results
    }

## Part 1.2 & 1.3: Head - Tail & Tail

In [21]:
# Run head-tail truncation experiment.

head_tail_results = run_experiment(
    strategy="head_tail",
    train_df=train_df,
    val_df=val_df,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    seed=SEED,
    num_labels=num_labels,
    label_names=label_names,
    project_name=PROJECT_NAME
)

# Run tail truncation experiment.

tail_results = run_experiment(
    strategy="tail",
    train_df=train_df,
    val_df=val_df,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    seed=SEED,
    num_labels=num_labels,
    label_names=label_names,
    project_name=PROJECT_NAME
)


===== RUNNING EXPERIMENT: roberta_head_tail_512 =====
Train max length after truncation: 512
Val max length after truncation: 512


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation metrics:
{'eval_loss': 0.42387107014656067, 'eval_accuracy': 0.9020618556701031, 'eval_macro_f1': 0.6053442580597538, 'eval_weighted_f1': 0.8949187580010628, 'eval_precision_class_0': 0.9125, 'eval_recall_class_0': 0.8902439024390244, 'eval_f1_class_0': 0.9012345679012346, 'eval_precision_class_1': 0.8947368421052632, 'eval_recall_class_1': 0.9357798165137615, 'eval_f1_class_1': 0.9147982062780269, 'eval_precision_class_2': 0.0, 'eval_recall_class_2': 0.0, 'eval_f1_class_2': 0.0, 'eval_runtime': 1.3433, 'eval_samples_per_second': 144.419, 'eval_steps_per_second': 9.678, 'epoch': 3.0}

===== TRAIN CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.96      0.91      0.94       326
     neutral       0.91      0.97      0.94       433
    positive       0.00      0.00      0.00        14

    accuracy                           0.93       773
   macro avg       0.62      0.63      0.63       773
weighted avg       0.91      0


===== VAL CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.91      0.89      0.90        82
     neutral       0.89      0.94      0.91       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.90       194
   macro avg       0.60      0.61      0.61       194
weighted avg       0.89      0.90      0.89       194


===== VAL CONFUSION MATRIX =====
[[ 73   9   0]
 [  7 102   0]
 [  0   3   0]]


eval/accuracy,▁▃██
eval/f1_class_0,▁▃██
eval/f1_class_1,▁▂██
eval/f1_class_2,▁▁▁▁
eval/loss,▆▁██
eval/macro_f1,▁▃██
eval/precision_class_0,█▁▂▂
eval/precision_class_1,▁▄██
eval/precision_class_2,▁▁▁▁
eval/recall_class_0,▁▅██
+51,...


## Part 1 - Overall Comparison

In [22]:
# =========================================================
# Truncation Strategy Comparison
# =========================================================
# Compare validation results across truncation strategies.


# Collect validation metrics for each strategy.
comparison_rows = []

comparison_rows.append({
    "strategy": head_results["strategy"],
    "run_name": head_results["run_name"],
    "val_macro_f1": head_results["val_results"]["metrics"]["val_macro_f1"],
    "val_accuracy": head_results["val_results"]["metrics"]["val_accuracy"],
})

comparison_rows.append({
    "strategy": head_tail_results["strategy"],
    "run_name": head_tail_results["run_name"],
    "val_macro_f1": head_tail_results["val_results"]["metrics"]["val_macro_f1"],
    "val_accuracy": head_tail_results["val_results"]["metrics"]["val_accuracy"],
})

comparison_rows.append({
    "strategy": tail_results["strategy"],
    "run_name": tail_results["run_name"],
    "val_macro_f1": tail_results["val_results"]["metrics"]["val_macro_f1"],
    "val_accuracy": tail_results["val_results"]["metrics"]["val_accuracy"],
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)

comparison_df

,strategy,run_name,val_macro_f1,val_accuracy
0,head_tail,roberta_head_tail_512,0.608671,0.907216
1,tail,roberta_tail_512,0.605344,0.902062
2,head,roberta_head_512,0.590453,0.881443


In [23]:
# Compare class-level validation F1 across strategies.

per_class_rows = []

for result_obj in [head_results, head_tail_results, tail_results]:
    row = {
        "strategy": result_obj["strategy"],
        "run_name": result_obj["run_name"],
    }

    for i, label_name in enumerate(label_names):
        row[f"val_f1_{label_name}"] = result_obj["val_results"]["metrics"][f"val_f1_class_{i}"]

    per_class_rows.append(row)

per_class_df = pd.DataFrame(per_class_rows)
per_class_df

,strategy,run_name,val_f1_negative,val_f1_neutral,val_f1_positive
0,head,roberta_head_512,0.871795,0.899563,0.0
1,head_tail,roberta_head_tail_512,0.905660,0.920354,0.0
2,tail,roberta_tail_512,0.901235,0.914798,0.0


## Part 1 - Truncation Result

The truncation experiments show only small differences across strategies.

- Head-tail truncation achieves the highest validation macro F1 (0.609)
- Tail truncation performs very similarly (0.605)
- Head truncation performs slightly worse (0.590)

At the class level:

- Negative and neutral classes are learned well across all strategies
- The model completely fails to detect the positive class (F1 = 0.0)

This indicates that truncation choice has only a limited impact on overall performance,
and does not address the core issue of minority-class learning.

Although head-tail truncation achieves the best validation score,
the difference with tail truncation is marginal.

Tail truncation is selected for the next experiments because:
- it performs competitively
- and it aligns with the assumption that sentiment is often expressed
  toward the end of customer conversations

# Part 2: Imbalance Handling Experiments

The truncation experiments show that text selection has only a limited
impact on performance, while the model still fails to learn the positive class.

This indicates that the main challenge is class imbalance rather than
input representation.

From this point onward, the text configuration is fixed:
- truncation strategy: tail
- maximum sequence length: 512
- model backbone: roberta-base

The following experiments focus on imbalance-handling methods:

- loss-level methods (class-weighted loss, focal loss)
- data-level methods (oversampling, under-sampling)

The goal is to evaluate whether these approaches improve the model’s
ability to learn the minority positive class.

In [24]:
# Rebuild datasets using the selected tail configuration.

BEST_TRUNCATION_STRATEGY = "tail"
BEST_MAX_LENGTH = 512

print("Selected truncation strategy:", BEST_TRUNCATION_STRATEGY)
print("Selected max length:", BEST_MAX_LENGTH)

Selected truncation strategy: tail
Selected max length: 512


In [25]:
# Tokenize tail-truncated text.

train_tail = apply_truncation(train_df, BEST_TRUNCATION_STRATEGY, tokenizer, BEST_MAX_LENGTH)
val_tail = apply_truncation(val_df, BEST_TRUNCATION_STRATEGY, tokenizer, BEST_MAX_LENGTH)

train_tail_hf = Dataset.from_pandas(train_tail[["model_text", "label"]], preserve_index=False)
val_tail_hf = Dataset.from_pandas(val_tail[["model_text", "label"]], preserve_index=False)

def tokenize_batch_tail(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=BEST_MAX_LENGTH,
        padding=False
    )

train_tail_hf = train_tail_hf.map(tokenize_batch_tail, batched=True)
val_tail_hf = val_tail_hf.map(tokenize_batch_tail, batched=True)

train_tail_hf = train_tail_hf.rename_column("label", "labels")
val_tail_hf = val_tail_hf.rename_column("label", "labels")

columns_to_keep = ["input_ids", "attention_mask", "labels"]

train_tail_hf.set_format(type="torch", columns=columns_to_keep)
val_tail_hf.set_format(type="torch", columns=columns_to_keep)

print(train_tail_hf)
print(val_tail_hf)

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Dataset({
    features: ['model_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 773
})
Dataset({
    features: ['model_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 194
})


## Part 2.1 - Class-Weighted Loss

Class imbalance is first addressed using a loss-level approach.

Class weights are computed from the training data and applied to
the cross-entropy loss function.

In [26]:
# Compute class weights based on training label distribution.

from sklearn.utils.class_weight import compute_class_weight

class_labels = np.sort(train_df["label"].unique())
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=train_df["label"].values
)

class_weights = torch.tensor(class_weights_np, dtype=torch.float)

print("Class labels:", class_labels)
print("Class weights:", class_weights_np)

class_weight_df = pd.DataFrame({
    "label_id": class_labels,
    "label_name": label_names,
    "class_weight": class_weights_np
})
class_weight_df

Class labels: [0 1 2]
Class weights: [ 0.79038855  0.59507313 18.4047619 ]


,label_id,label_name,class_weight
0,0,negative,0.790389
1,1,neutral,0.595073
2,2,positive,18.404762


In [27]:
# Custom Trainer using class-weighted cross-entropy loss.

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs.get("input_ids"),
            attention_mask=inputs.get("attention_mask"),
            labels=None
        )
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [28]:
# Initialize experiment for class-weighted training.
WEIGHTED_RUN_NAME = f"roberta_tail_{BEST_MAX_LENGTH}_classweights"

wandb.init(
    project=PROJECT_NAME,
    name=WEIGHTED_RUN_NAME,
    config={
        "seed": SEED,
        "model_name": MODEL_NAME,
        "max_length": BEST_MAX_LENGTH,
        "truncation_strategy": BEST_TRUNCATION_STRATEGY,
        "imbalance_method": "class_weighted_loss",
        "class_weights": class_weights_np.tolist(),
        "num_labels": num_labels,
        "train_size": len(train_df),
        "val_size": len(val_df),
    }
)

reset_all_seeds(SEED)

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
weighted_model.resize_token_embeddings(len(tokenizer))

weighted_training_args = TrainingArguments(
    output_dir=f"./outputs/{WEIGHTED_RUN_NAME}_new",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=WEIGHTED_RUN_NAME,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0
)

weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=weighted_training_args,
    train_dataset=train_tail_hf,
    eval_dataset=val_tail_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

print("Run name:", WEIGHTED_RUN_NAME)
print("Class weights:", class_weights_np)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Run name: roberta_tail_512_classweights
Class weights: [ 0.79038855  0.59507313 18.4047619 ]


In [29]:
# Train model with class-weighted loss.

weighted_train_result = weighted_trainer.train()
weighted_val_metrics = weighted_trainer.evaluate()

print("\nValidation metrics:")
print(weighted_val_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2
1,0.928069,0.847189,0.809278,0.536100,0.797781,0.900000,0.658537,0.760563,0.768657,0.944954,0.847737,0.000000,0.000000,0.000000
2,0.661067,0.581410,0.881443,0.589598,0.873189,0.955882,0.792683,0.866667,0.841270,0.972477,0.902128,0.000000,0.000000,0.000000
3,0.344832,0.655047,0.891753,0.598239,0.884661,0.900000,0.878049,0.888889,0.885965,0.926606,0.905830,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation metrics:
{'eval_loss': 0.6550469994544983, 'eval_accuracy': 0.8917525773195877, 'eval_macro_f1': 0.5982394951004816, 'eval_weighted_f1': 0.8846614169992654, 'eval_precision_class_0': 0.9, 'eval_recall_class_0': 0.8780487804878049, 'eval_f1_class_0': 0.8888888888888888, 'eval_precision_class_1': 0.8859649122807017, 'eval_recall_class_1': 0.926605504587156, 'eval_f1_class_1': 0.905829596412556, 'eval_precision_class_2': 0.0, 'eval_recall_class_2': 0.0, 'eval_f1_class_2': 0.0, 'eval_runtime': 1.3409, 'eval_samples_per_second': 144.681, 'eval_steps_per_second': 9.695, 'epoch': 3.0}


In [30]:
# Evaluate model performance on train and validation sets.
weighted_train_results = evaluate_split(weighted_trainer, train_tail_hf, "train", label_names)
weighted_val_results = evaluate_split(weighted_trainer, val_tail_hf, "val", label_names)

class_weighted_results = {
    "run_name": WEIGHTED_RUN_NAME,
    "strategy": "tail_classweights",
    "train_results": weighted_train_results,
    "val_results": weighted_val_results
}

wandb.finish()


===== TRAIN CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.93      0.92      0.92       326
     neutral       0.91      0.95      0.93       433
    positive       0.00      0.00      0.00        14

    accuracy                           0.92       773
   macro avg       0.61      0.62      0.62       773
weighted avg       0.90      0.92      0.91       773


===== TRAIN CONFUSION MATRIX =====
[[299  27   0]
 [ 22 411   0]
 [  0  14   0]]



===== VAL CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89        82
     neutral       0.89      0.93      0.91       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.89       194
   macro avg       0.60      0.60      0.60       194
weighted avg       0.88      0.89      0.88       194


===== VAL CONFUSION MATRIX =====
[[ 72  10   0]
 [  8 101   0]
 [  0   3   0]]


eval/accuracy,▁▇██
eval/f1_class_0,▁▇██
eval/f1_class_1,▁███
eval/f1_class_2,▁▁▁▁
eval/loss,█▁▃▃
eval/macro_f1,▁▇██
eval/precision_class_0,▁█▁▁
eval/precision_class_1,▁▅██
eval/precision_class_2,▁▁▁▁
eval/recall_class_0,▁▅██
+51,...


In [31]:
# Compare the results with the baseline
imbalance_comparison_rows = [
    {
        "model": "tail_baseline",
        "run_name": tail_results["run_name"],
        "val_macro_f1": tail_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": tail_results["val_results"]["metrics"]["val_accuracy"],
    },
    {
        "model": "tail_classweights",
        "run_name": class_weighted_results["run_name"],
        "val_macro_f1": class_weighted_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": class_weighted_results["val_results"]["metrics"]["val_accuracy"],
    }
]

imbalance_comparison_df = pd.DataFrame(imbalance_comparison_rows)
imbalance_comparison_df

,model,run_name,val_macro_f1,val_accuracy
0,tail_baseline,roberta_tail_512,0.605344,0.902062
1,tail_classweights,roberta_tail_512_classweights,0.598239,0.891753


In [32]:
per_class_compare_rows = []

for result_obj, model_name in [
    (tail_results, "tail_baseline"),
    (class_weighted_results, "tail_classweights")
]:
    row = {
        "model": model_name,
        "run_name": result_obj["run_name"]
    }

    for i, label_name in enumerate(label_names):
        row[f"val_f1_{label_name}"] = result_obj["val_results"]["metrics"][f"val_f1_class_{i}"]

    per_class_compare_rows.append(row)

per_class_compare_df = pd.DataFrame(per_class_compare_rows)
per_class_compare_df

,model,run_name,val_f1_negative,val_f1_neutral,val_f1_positive
0,tail_baseline,roberta_tail_512,0.901235,0.914798,0.0
1,tail_classweights,roberta_tail_512_classweights,0.888889,0.905830,0.0


## Part 2.1 - Result of Class-Weighted Loss

Applying class-weighted cross-entropy does not improve performance.

- Validation macro F1 decreases from 0.605 to 0.598
- Validation accuracy also decreases slightly

At the class level:

- Performance on negative and neutral classes drops slightly
- The model still fails to detect the positive class (F1 = 0.0)

This indicates that reweighting the loss alone is not sufficient
to address the imbalance problem.

## Part 2.2 - Focal Loss

Focal loss is applied as an alternative loss-level approach to address class imbalance.

Unlike standard cross-entropy, focal loss reduces the contribution of easy examples and focuses training on harder, misclassified samples.

In [33]:
# Define focal loss to focus on hard examples.

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, labels):
        ce_loss = self.ce(logits, labels)   # shape: [batch_size]
        pt = torch.exp(-ce_loss)

        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)   # move to same device first
            alpha_t = alpha[labels]                # class weight for each sample
            loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        else:
            loss = (1 - pt) ** self.gamma * ce_loss

        return loss.mean()

# Custom Trainer using focal loss.
class FocalTrainer(Trainer):
    def __init__(self, focal_loss=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = focal_loss

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs.get("input_ids"),
            attention_mask=inputs.get("attention_mask"),
            labels=None
        )
        logits = outputs.get("logits")

        loss = self.focal_loss(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Initialize experiment tracking for focal loss.
FOCAL_RUN_NAME = f"roberta_tail_{BEST_MAX_LENGTH}_focal"
FOCAL_GAMMA = 2.0

focal_loss_fn = FocalLoss(
    alpha=class_weights,
    gamma=FOCAL_GAMMA
)

wandb.init(
    project=PROJECT_NAME,
    name=FOCAL_RUN_NAME,
    config={
        "seed": SEED,
        "model_name": MODEL_NAME,
        "max_length": BEST_MAX_LENGTH,
        "truncation_strategy": BEST_TRUNCATION_STRATEGY,
        "imbalance_method": "focal_loss",
        "gamma": FOCAL_GAMMA,
        "alpha": class_weights_np.tolist(),
        "num_labels": num_labels,
        "train_size": len(train_df),
        "val_size": len(val_df),
    }
)

reset_all_seeds(SEED)

focal_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

focal_model.resize_token_embeddings(len(tokenizer))

focal_training_args = TrainingArguments(
    output_dir=f"./outputs/{FOCAL_RUN_NAME}_new",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=FOCAL_RUN_NAME,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0
)

focal_trainer = FocalTrainer(
    model=focal_model,
    args=focal_training_args,
    train_dataset=train_tail_hf,
    eval_dataset=val_tail_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    focal_loss=focal_loss_fn
)

print("Run name:", FOCAL_RUN_NAME)
print("Gamma:", FOCAL_GAMMA)
print("Alpha (class weights):", class_weights_np)

# Train model with focal loss.
focal_train_result = focal_trainer.train()
focal_val_metrics = focal_trainer.evaluate()

print("\nValidation metrics:")
print(focal_val_metrics)

# Evaluate focal loss model.
focal_train_results = evaluate_split(focal_trainer, train_tail_hf, "train", label_names)
focal_val_results = evaluate_split(focal_trainer, val_tail_hf, "val", label_names)

focal_results = {
    "run_name": FOCAL_RUN_NAME,
    "strategy": "tail_focal",
    "train_results": focal_train_results,
    "val_results": focal_val_results
}

wandb.finish()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Run name: roberta_tail_512_focal
Gamma: 2.0
Alpha (class weights): [ 0.79038855  0.59507313 18.4047619 ]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2
1,1.324177,0.927935,0.865979,0.579201,0.858029,0.915493,0.792683,0.849673,0.837398,0.944954,0.887931,0.000000,0.000000,0.000000
2,0.635254,0.457332,0.865979,0.578857,0.857726,0.927536,0.780488,0.847682,0.832000,0.954128,0.888889,0.000000,0.000000,0.000000
3,0.119035,0.426374,0.912371,0.612449,0.905176,0.925000,0.902439,0.913580,0.903509,0.944954,0.923767,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation metrics:
{'eval_loss': 0.42637404799461365, 'eval_accuracy': 0.9123711340206185, 'eval_macro_f1': 0.612449021019026, 'eval_weighted_f1': 0.9051760990028601, 'eval_precision_class_0': 0.925, 'eval_recall_class_0': 0.9024390243902439, 'eval_f1_class_0': 0.9135802469135802, 'eval_precision_class_1': 0.9035087719298246, 'eval_recall_class_1': 0.944954128440367, 'eval_f1_class_1': 0.9237668161434978, 'eval_precision_class_2': 0.0, 'eval_recall_class_2': 0.0, 'eval_f1_class_2': 0.0, 'eval_runtime': 1.3302, 'eval_samples_per_second': 145.847, 'eval_steps_per_second': 9.773, 'epoch': 3.0}

===== TRAIN CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.96      0.92      0.94       326
     neutral       0.91      0.97      0.94       433
    positive       0.00      0.00      0.00        14

    accuracy                           0.93       773
   macro avg       0.62      0.63      0.62       773
weighted avg       0.91      0.93


===== VAL CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.93      0.90      0.91        82
     neutral       0.90      0.94      0.92       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.91       194
   macro avg       0.61      0.62      0.61       194
weighted avg       0.90      0.91      0.91       194


===== VAL CONFUSION MATRIX =====
[[ 74   8   0]
 [  6 103   0]
 [  0   3   0]]


eval/accuracy,▁▁██
eval/f1_class_0,▁▁██
eval/f1_class_1,▁▁██
eval/f1_class_2,▁▁▁▁
eval/loss,█▁▁▁
eval/macro_f1,▁▁██
eval/precision_class_0,▁█▇▇
eval/precision_class_1,▂▁██
eval/precision_class_2,▁▁▁▁
eval/recall_class_0,▂▁██
+51,...


In [34]:
# Compare results with the baseline
focal_comparison_rows = [
    {
        "model": "tail_baseline",
        "run_name": tail_results["run_name"],
        "val_macro_f1": tail_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": tail_results["val_results"]["metrics"]["val_accuracy"],
    },
    {
        "model": "tail_classweights",
        "run_name": class_weighted_results["run_name"],
        "val_macro_f1": class_weighted_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": class_weighted_results["val_results"]["metrics"]["val_accuracy"],
    },
    {
        "model": "tail_focal",
        "run_name": focal_results["run_name"],
        "val_macro_f1": focal_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": focal_results["val_results"]["metrics"]["val_accuracy"],
    }
]

focal_comparison_df = pd.DataFrame(focal_comparison_rows)
focal_comparison_df

,model,run_name,val_macro_f1,val_accuracy
0,tail_baseline,roberta_tail_512,0.605344,0.902062
1,tail_classweights,roberta_tail_512_classweights,0.598239,0.891753
2,tail_focal,roberta_tail_512_focal,0.612449,0.912371


In [35]:
focal_per_class_rows = []

for result_obj, model_name in [
    (tail_results, "tail_baseline"),
    (class_weighted_results, "tail_classweights"),
    (focal_results, "tail_focal")
]:
    row = {
        "model": model_name,
        "run_name": result_obj["run_name"]
    }

    for i, label_name in enumerate(label_names):
        row[f"val_f1_{label_name}"] = result_obj["val_results"]["metrics"][f"val_f1_class_{i}"]

    focal_per_class_rows.append(row)

focal_per_class_df = pd.DataFrame(focal_per_class_rows)
focal_per_class_df

,model,run_name,val_f1_negative,val_f1_neutral,val_f1_positive
0,tail_baseline,roberta_tail_512,0.901235,0.914798,0.0
1,tail_classweights,roberta_tail_512_classweights,0.888889,0.905830,0.0
2,tail_focal,roberta_tail_512_focal,0.913580,0.923767,0.0


## Part 2.2 - Result of Focal Loss

Focal loss improves overall validation performance:

- Validation macro F1 increases from 0.605 to 0.612

At the class level:

- Performance on negative and neutral classes improves
- The model still fails to detect the positive class (F1 = 0.0)

This indicates that focal loss helps the model focus on harder examples,
but is still not sufficient to address the imbalance problem.

Despite improvements in overall metrics, the model continues to struggle
with learning meaningful patterns for the positive class.

## Part 2.3: Oversampling Experiments

Loss-level methods such as class-weighted cross-entropy and focal loss do not improve detection of the positive class.

A data-level approach is therefore considered: **oversampling**.

Instead of fully balancing all classes, a more conservative strategy is applied by oversampling only the positive class. This increases the model’s exposure to minority-class examples while limiting the risk of severe overfitting.

In [40]:
# Get the encoded label id for the positive class.
positive_label_id = label_map_df.loc[
    label_map_df["customer_sentiment"].str.lower() == "positive", "label"
].iloc[0]

print("Positive label id:", positive_label_id)

Positive label id: 2


In [41]:
# Oversample only the positive class by a factor of 3.
# Only the training set is oversampled. Validation and test sets remain unchanged
# so that evaluation remains fair and comparable with previous experiments.

from sklearn.utils import resample

OVERSAMPLE_FACTOR = 3

print("Original training class distribution:")
print(train_df["label"].value_counts().sort_index())

train_majority_part = train_df[train_df["label"] != positive_label_id].copy()
train_positive_part = train_df[train_df["label"] == positive_label_id].copy()

reset_all_seeds(SEED)

# Randomly duplicate positive samples with replacement.
train_positive_oversampled = resample(
    train_positive_part,
    replace=True,
    n_samples=len(train_positive_part) * OVERSAMPLE_FACTOR,
    random_state=SEED
)

# Combine oversampled positive rows with all other rows.
train_os_3x = pd.concat([train_majority_part, train_positive_oversampled], axis=0)
train_os_3x = train_os_3x.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nOversampled training class distribution (positive x3):")
print(train_os_3x["label"].value_counts().sort_index())

Original training class distribution:
label
0    326
1    433
2     14
Name: count, dtype: int64

Oversampled training class distribution (positive x3):
label
0    326
1    433
2     42
Name: count, dtype: int64


In [42]:
# Apply the selected tail truncation to oversampled training data.
train_os_3x_tail = apply_truncation(train_os_3x, BEST_TRUNCATION_STRATEGY, tokenizer, BEST_MAX_LENGTH)

# Convert oversampled training data to Hugging Face format.
train_os_3x_hf = Dataset.from_pandas(
    train_os_3x_tail[["model_text", "label"]],
    preserve_index=False
)

# Tokenize oversampled training text.
train_os_3x_hf = train_os_3x_hf.map(tokenize_batch_tail, batched=True)
train_os_3x_hf = train_os_3x_hf.rename_column("label", "labels")
train_os_3x_hf.set_format(type="torch", columns=columns_to_keep)

print(train_os_3x_hf)
print(val_tail_hf)

Map:   0%|          | 0/801 [00:00<?, ? examples/s]

Dataset({
    features: ['model_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 801
})
Dataset({
    features: ['model_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 194
})


In [43]:
# Initialize experiment tracking for positive-class oversampling.
OVERSAMPLE_RUN_NAME = f"roberta_tail_{BEST_MAX_LENGTH}_oversample_pos3"

wandb.init(
    project=PROJECT_NAME,
    name=OVERSAMPLE_RUN_NAME,
    config={
        "seed": SEED,
        "model_name": MODEL_NAME,
        "max_length": BEST_MAX_LENGTH,
        "truncation_strategy": BEST_TRUNCATION_STRATEGY,
        "imbalance_method": "oversample_positive_only",
        "oversample_factor": OVERSAMPLE_FACTOR,
        "num_labels": num_labels,
        "train_size_original": len(train_df),
        "train_size_oversampled": len(train_os_3x),
        "val_size": len(val_df),
    }
)

# Reset seeds and load a fresh model for oversampling.
reset_all_seeds(SEED)

oversample_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
oversample_model.resize_token_embeddings(len(tokenizer))

# Use the same training setup for fair comparison.
oversample_training_args = TrainingArguments(
    output_dir=f"./outputs/{OVERSAMPLE_RUN_NAME}_new",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=OVERSAMPLE_RUN_NAME,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0
)

oversample_trainer = Trainer(
    model=oversample_model,
    args=oversample_training_args,
    train_dataset=train_os_3x_hf,
    eval_dataset=val_tail_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Run name:", OVERSAMPLE_RUN_NAME)
print("Oversample factor:", OVERSAMPLE_FACTOR)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Run name: roberta_tail_512_oversample_pos3
Oversample factor: 3


In [44]:
# Train baseline RoBERTa on oversampled training data.
oversample_train_result = oversample_trainer.train()
oversample_val_metrics = oversample_trainer.evaluate()

print("\nValidation metrics:")
print(oversample_val_metrics)

# Evaluate train and validation splits in detail.
oversample_train_results = evaluate_split(oversample_trainer, train_os_3x_hf, "train", label_names)
oversample_val_results = evaluate_split(oversample_trainer, val_tail_hf, "val", label_names)

oversample_results = {
    "run_name": OVERSAMPLE_RUN_NAME,
    "strategy": "tail_oversample_pos3",
    "train_results": oversample_train_results,
    "val_results": oversample_val_results
}

wandb.finish()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2
1,0.530436,0.461020,0.850515,0.571364,0.844000,0.785714,0.939024,0.855556,0.916667,0.807339,0.858537,0.000000,0.000000,0.000000
2,0.293583,0.307722,0.891753,0.598239,0.884661,0.900000,0.878049,0.888889,0.885965,0.926606,0.905830,0.000000,0.000000,0.000000
3,0.195523,0.361072,0.886598,0.598437,0.884226,0.911392,0.878049,0.894410,0.884956,0.917431,0.900901,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation metrics:
{'eval_loss': 0.3610718846321106, 'eval_accuracy': 0.8865979381443299, 'eval_macro_f1': 0.5984369462630332, 'eval_weighted_f1': 0.8842258407475799, 'eval_precision_class_0': 0.9113924050632911, 'eval_recall_class_0': 0.8780487804878049, 'eval_f1_class_0': 0.8944099378881988, 'eval_precision_class_1': 0.8849557522123894, 'eval_recall_class_1': 0.9174311926605505, 'eval_f1_class_1': 0.9009009009009009, 'eval_precision_class_2': 0.0, 'eval_recall_class_2': 0.0, 'eval_f1_class_2': 0.0, 'eval_runtime': 1.3159, 'eval_samples_per_second': 147.426, 'eval_steps_per_second': 9.879, 'epoch': 3.0}

===== TRAIN CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.96      0.93      0.94       326
     neutral       0.93      0.95      0.94       433
    positive       0.80      0.86      0.83        42

    accuracy                           0.94       801
   macro avg       0.90      0.91      0.90       801
weighted avg       


===== VAL CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.91      0.88      0.89        82
     neutral       0.88      0.92      0.90       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.89       194
   macro avg       0.60      0.60      0.60       194
weighted avg       0.88      0.89      0.88       194


===== VAL CONFUSION MATRIX =====
[[ 72  10   0]
 [  7 100   2]
 [  0   3   0]]


eval/accuracy,▁█▇▇
eval/f1_class_0,▁▇██
eval/f1_class_1,▁█▇▇
eval/f1_class_2,▁▁▁▁
eval/loss,█▁▃▃
eval/macro_f1,▁███
eval/precision_class_0,▁▇██
eval/precision_class_1,█▁▁▁
eval/precision_class_2,▁▁▁▁
eval/recall_class_0,█▁▁▁
+51,...


In [45]:
# Compare oversampling with earlier imbalance methods.
oversample_comparison_rows = [
    {
        "model": "tail_baseline",
        "run_name": tail_results["run_name"],
        "val_macro_f1": tail_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": tail_results["val_results"]["metrics"]["val_accuracy"],
    },
    {
        "model": "tail_classweights",
        "run_name": class_weighted_results["run_name"],
        "val_macro_f1": class_weighted_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": class_weighted_results["val_results"]["metrics"]["val_accuracy"],
    },
    {
        "model": "tail_focal",
        "run_name": focal_results["run_name"],
        "val_macro_f1": focal_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": focal_results["val_results"]["metrics"]["val_accuracy"],
    },
    {
        "model": "tail_oversample_pos3",
        "run_name": oversample_results["run_name"],
        "val_macro_f1": oversample_results["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": oversample_results["val_results"]["metrics"]["val_accuracy"],
    }
]

oversample_comparison_df = pd.DataFrame(oversample_comparison_rows)
oversample_comparison_df

,model,run_name,val_macro_f1,val_accuracy
0,tail_baseline,roberta_tail_512,0.605344,0.902062
1,tail_classweights,roberta_tail_512_classweights,0.598239,0.891753
2,tail_focal,roberta_tail_512_focal,0.612449,0.912371
3,tail_oversample_pos3,roberta_tail_512_oversample_pos3,0.598437,0.886598


In [46]:
# Compare class-level validation F1 scores across methods.
oversample_per_class_rows = []

for result_obj, model_name in [
    (tail_results, "tail_baseline"),
    (class_weighted_results, "tail_classweights"),
    (focal_results, "tail_focal"),
    (oversample_results, "tail_oversample_pos3")
]:
    row = {
        "model": model_name,
        "run_name": result_obj["run_name"]
    }

    for i, label_name in enumerate(label_names):
        row[f"val_f1_{label_name}"] = result_obj["val_results"]["metrics"][f"val_f1_class_{i}"]

    oversample_per_class_rows.append(row)

oversample_per_class_df = pd.DataFrame(oversample_per_class_rows)
oversample_per_class_df

,model,run_name,val_f1_negative,val_f1_neutral,val_f1_positive
0,tail_baseline,roberta_tail_512,0.901235,0.914798,0.0
1,tail_classweights,roberta_tail_512_classweights,0.888889,0.905830,0.0
2,tail_focal,roberta_tail_512_focal,0.913580,0.923767,0.0
3,tail_oversample_pos3,roberta_tail_512_oversample_pos3,0.894410,0.900901,0.0


## Part 2.3 - Result of Oversampling (Positive x3)

Oversampling the positive class leads to a clear change in model behavior.

- The model begins to learn and predict the positive class
- Training performance improves significantly (positive F1 ≈ 0.83)

However, validation performance does not improve:

- Validation macro F1 decreases from 0.612 to 0.598
- The model still fails to detect the positive class on validation (F1 = 0.0)

This indicates that oversampling successfully increases exposure to the
minority class, but does not generalize well.

Overall, oversampling helps the model learn the positive class at the
training level, but is not sufficient to achieve reliable validation performance.

## Part 2.4: Under-Sampling Experiment

Under-sampling is applied as an alternative data-level approach.

Instead of increasing the number of minority samples, this method reduces
the size of majority classes. This aims to balance the dataset while avoiding
duplicate samples.

In [47]:
# Compute positive class size and define target size for other classes.

positive_label_id = label_map_df.loc[
    label_map_df["customer_sentiment"].str.lower() == "positive", "label"
].iloc[0]

positive_count = (train_df["label"] == positive_label_id).sum()
target_majority_count = positive_count * 3

print("Original training class distribution:")
print(train_df["label"].value_counts().sort_index())

reset_all_seeds(SEED)

# Downsample majority classes while keeping all positive samples.
undersampled_parts = []

for label in sorted(train_df["label"].unique()):
    df_label = train_df[train_df["label"] == label].copy()

    if label == positive_label_id:
        df_sampled = df_label
    else:
        n_target = min(len(df_label), target_majority_count)
        df_sampled = resample(
            df_label,
            replace=False,
            n_samples=n_target,
            random_state=SEED
        )

    undersampled_parts.append(df_sampled)

# Combine sampled subsets and shuffle the dataset.
train_us = pd.concat(undersampled_parts, axis=0)
train_us = train_us.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nUnder-sampled training class distribution:")
print(train_us["label"].value_counts().sort_index())

Original training class distribution:
label
0    326
1    433
2     14
Name: count, dtype: int64

Under-sampled training class distribution:
label
0    42
1    42
2    14
Name: count, dtype: int64


In [48]:
# Apply tail truncation and prepare Hugging Face dataset.
train_us_tail = apply_truncation(train_us, BEST_TRUNCATION_STRATEGY, tokenizer, BEST_MAX_LENGTH)

train_us_hf = Dataset.from_pandas(
    train_us_tail[["model_text", "label"]],
    preserve_index=False
)

train_us_hf = train_us_hf.map(tokenize_batch_tail, batched=True)
train_us_hf = train_us_hf.rename_column("label", "labels")
train_us_hf.set_format(type="torch", columns=columns_to_keep)

print(train_us_hf)

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Dataset({
    features: ['model_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 98
})


In [49]:
# Initialize experiment for under-sampling.
UNDERSAMPLE_RUN_NAME = f"roberta_tail_{BEST_MAX_LENGTH}_undersample"

wandb.init(
    project=PROJECT_NAME,
    name=UNDERSAMPLE_RUN_NAME,
    config={
        "seed": SEED,
        "model_name": MODEL_NAME,
        "max_length": BEST_MAX_LENGTH,
        "truncation_strategy": BEST_TRUNCATION_STRATEGY,
        "imbalance_method": "undersampling",
        "target_majority_count": target_majority_count,
        "num_labels": num_labels,
        "train_size_original": len(train_df),
        "train_size_undersampled": len(train_us),
        "val_size": len(val_df)
    }
)

reset_all_seeds(SEED)

undersample_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
undersample_model.resize_token_embeddings(len(tokenizer))

undersample_training_args = TrainingArguments(
    output_dir=f"./outputs/{UNDERSAMPLE_RUN_NAME}_new",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=UNDERSAMPLE_RUN_NAME,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0
)

undersample_trainer = Trainer(
    model=undersample_model,
    args=undersample_training_args,
    train_dataset=train_us_hf,
    eval_dataset=val_tail_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Run name:", UNDERSAMPLE_RUN_NAME)
print("Target majority count:", target_majority_count)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Run name: roberta_tail_512_undersample
Target majority count: 42


In [50]:
# Train model on under-sampled data.
undersample_train_result = undersample_trainer.train()
undersample_val_metrics = undersample_trainer.evaluate()

print("\nValidation metrics:")
print(undersample_val_metrics)

# Evaluate model performance.
undersample_train_results = evaluate_split(undersample_trainer, train_us_hf, "train", label_names)
undersample_val_results = evaluate_split(undersample_trainer, val_tail_hf, "val", label_names)

undersample_results = {
    "run_name": UNDERSAMPLE_RUN_NAME,
    "strategy": "tail_undersample",
    "train_results": undersample_train_results,
    "val_results": undersample_val_results
}

wandb.finish()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2
1,No log,0.947684,0.422680,0.198068,0.251158,0.422680,1.000000,0.594203,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1.055444,0.821159,0.654639,0.438703,0.643951,0.573770,0.853659,0.686275,0.791667,0.522936,0.629834,0.000000,0.000000,0.000000
3,1.055444,0.773714,0.664948,0.440417,0.657790,0.631579,0.585366,0.607595,0.686441,0.743119,0.713656,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation metrics:
{'eval_loss': 0.7737138867378235, 'eval_accuracy': 0.6649484536082474, 'eval_macro_f1': 0.4404171081246863, 'eval_weighted_f1': 0.6577903663176968, 'eval_precision_class_0': 0.631578947368421, 'eval_recall_class_0': 0.5853658536585366, 'eval_f1_class_0': 0.6075949367088608, 'eval_precision_class_1': 0.6864406779661016, 'eval_recall_class_1': 0.7431192660550459, 'eval_f1_class_1': 0.7136563876651982, 'eval_precision_class_2': 0.0, 'eval_recall_class_2': 0.0, 'eval_f1_class_2': 0.0, 'eval_runtime': 1.3127, 'eval_samples_per_second': 147.79, 'eval_steps_per_second': 9.903, 'epoch': 3.0}

===== TRAIN CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.71      0.60      0.65        42
     neutral       0.54      0.81      0.65        42
    positive       0.00      0.00      0.00        14

    accuracy                           0.60        98
   macro avg       0.42      0.47      0.43        98
weighted avg       0.


===== VAL CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    negative       0.63      0.59      0.61        82
     neutral       0.69      0.74      0.71       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.66       194
   macro avg       0.44      0.44      0.44       194
weighted avg       0.65      0.66      0.66       194


===== VAL CONFUSION MATRIX =====
[[48 34  0]
 [28 81  0]
 [ 0  3  0]]


eval/accuracy,▁███
eval/f1_class_0,▁█▂▂
eval/f1_class_1,▁▇██
eval/f1_class_2,▁▁▁▁
eval/loss,█▃▁▁
eval/macro_f1,▁███
eval/precision_class_0,▁▆██
eval/precision_class_1,▁█▇▇
eval/precision_class_2,▁▁▁▁
eval/recall_class_0,█▆▁▁
+51,...


In [51]:
# Compare under-sampling with previous methods.
balance_compare_rows = []

for result_obj, model_name in [
    (tail_results, "tail_baseline"),
    (class_weighted_results, "tail_classweights"),
    (focal_results, "tail_focal"),
    (oversample_results, "tail_oversample_pos3"),
    (undersample_results, "tail_undersample")
]:
    row = {
        "model": model_name,
        "run_name": result_obj["run_name"],
        "val_macro_f1": result_obj["val_results"]["metrics"]["val_macro_f1"],
        "val_accuracy": result_obj["val_results"]["metrics"]["val_accuracy"],
    }

    for i, label_name in enumerate(label_names):
        row[f"val_f1_{label_name}"] = result_obj["val_results"]["metrics"][f"val_f1_class_{i}"]

    balance_compare_rows.append(row)

balance_compare_df = pd.DataFrame(balance_compare_rows)
balance_compare_df

,model,run_name,val_macro_f1,val_accuracy,val_f1_negative,val_f1_neutral,val_f1_positive
0,tail_baseline,roberta_tail_512,0.605344,0.902062,0.901235,0.914798,0.0
1,tail_classweights,roberta_tail_512_classweights,0.598239,0.891753,0.888889,0.905830,0.0
2,tail_focal,roberta_tail_512_focal,0.612449,0.912371,0.913580,0.923767,0.0
3,tail_oversample_pos3,roberta_tail_512_oversample_pos3,0.598437,0.886598,0.894410,0.900901,0.0
4,tail_undersample,roberta_tail_512_undersample,0.440417,0.664948,0.607595,0.713656,0.0


## Part 2.4 - Result of Under-Sampling

Under-sampling significantly degrades model performance.

- Validation macro F1 drops from 0.605 (baseline) to 0.440
- Validation accuracy decreases sharply

At the class level:

- Performance on negative and neutral classes drops substantially
- The model still fails to detect the positive class (F1 = 0.0)

This indicates that reducing majority-class samples leads to loss of
important information and harms overall learning.

Under-sampling is not effective for this task.

## Part 2 - Overall Comparison

The imbalance-handling experiments show clear differences between methods.

- Class-weighted loss slightly degrades performance
- Focal loss improves overall metrics but still fails to detect the positive class
- Oversampling enables the model to learn the positive class in training,
  but does not improve validation performance
- Under-sampling significantly degrades performance

Overall, none of the methods successfully improve positive-class detection
on the validation set.

This suggests that the problem is not only class imbalance, but also limited
separability of the positive class in the current text representation.